# Lab 05 - Jobs y Workflows

**Objetivos:**
- Crear notebooks parametrizados para jobs
- Simular multi-task workflows
- Implementar error handling
- Retornar resultados estructurados

## Notebook 1: Ingesta (Task 1)

In [ ]:
# =============================================================================
# PARAMETRIZACIÓN CON WIDGETS - TASK 1: INGESTA
# =============================================================================
# Los Workflows en Databricks pasan parámetros entre tasks via widgets
# Esto hace que el mismo notebook sea reutilizable con diferentes valores

# Crear widgets para recibir parámetros del job
dbutils.widgets.text("process_date", "2026-05-22", "📅 Fecha de Proceso")
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "🌍 Environment")
dbutils.widgets.text("batch_size", "1000", "📦 Batch Size")

# Leer valores de los widgets
process_date = dbutils.widgets.get("process_date")
env = dbutils.widgets.get("env")
batch_size = int(dbutils.widgets.get("batch_size"))  # Convertir string a int

# 💡 EN UN WORKFLOW REAL:
# - El scheduler/orquestador (ej: Databricks Workflows, Airflow) pasa estos valores
# - Permite ejecutar el mismo pipeline con diferentes fechas/ambientes
# - Facilita reprocesamiento de fechas específicas

print(f"🚀 Task 1: Ingesta")
print(f"  Fecha: {process_date}")
print(f"  Env: {env}")
print(f"  Batch Size: {batch_size}")

In [ ]:
from pyspark.sql.functions import *
from datetime import datetime
import json  # Para serializar resultados estructurados
import random

# =============================================================================
# TRY-CATCH: ERROR HANDLING EN JOBS
# =============================================================================
# En workflows de producción, SIEMPRE manejar errores explícitamente
# Permite: logging detallado, notificaciones, recuperación automática

try:
    # =========================================================================
    # PASO 1: Simular ingesta de datos desde sistema fuente
    # =========================================================================
    df_source = spark.range(0, batch_size) \
        .withColumn("date", lit(process_date)) \
        .withColumn("value", rand() * 1000) \
        .withColumn("category", (rand() * 5).cast("int")) \
        .withColumn("extracted_at", current_timestamp())  # Timestamp de extracción
    
    # Nombre de tabla dinámico por ambiente y fecha
    table_name = f"jobs_{env}_bronze_{process_date.replace('-', '')}"
    output_path = f"dbfs:/user/hive/warehouse/{table_name}"
    
    # =========================================================================
    # PASO 2: Guardar en formato Delta
    # =========================================================================
    df_source.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)  # Usar saveAsTable para managed tables
    
    extract_count = df_source.count()
    print(f"✅ Ingesta: {extract_count} registros → {table_name}")
    
    # =========================================================================
    # PASO 3: Retornar resultado estructurado con dbutils.notebook.exit()
    # =========================================================================
    # ¡CLAVE! dbutils.notebook.exit() hace DOS cosas:
    # 1. Finaliza la ejecución del notebook (como return en una función)
    # 2. Retorna un valor al notebook/job llamador (para workflows multi-task)
    
    # 💡 BEST PRACTICES PARA JOBS:
    # 1. Siempre usar try-except con logging detallado
    # 2. Retornar resultados estructurados (JSON) con dbutils.notebook.exit()
    # 3. Incluir timestamps y métricas para observabilidad
    # 4. Paths dinámicos por ambiente y fecha (facilita testing y recovery)
    
    result = {
        "status": "SUCCESS",               # Estado de la tarea
        "task": "ingest",                  # Nombre de la tarea
        "records_ingested": extract_count, # Métricas de negocio
        "table_name": table_name,          # Tabla para siguiente task
        "timestamp": datetime.utcnow().isoformat()  # ISO 8601 timestamp
    }
    
    # Serializar resultado a JSON y retornar
    # La siguiente task en el workflow puede leer este resultado
    dbutils.notebook.exit(json.dumps(result))

# =============================================================================
# MANEJO DE ERRORES
# =============================================================================
except Exception as e:
    # Si algo falla, retornar error estructurado (NO dejar que crashee sin info)
    print(f"❌ Error: {e}")
    
    error_result = {
        "status": "FAILED",
        "task": "ingest",
        "error": str(e),                   # Mensaje de error para debugging
        "timestamp": datetime.utcnow().isoformat()
    }
    
    # dbutils.notebook.exit() con status FAILED
    # El workflow puede decidir: ¿reintentar? ¿notificar? ¿abortar?
    dbutils.notebook.exit(json.dumps(error_result))

## Notebook 2: Transformación (Task 2)

In [ ]:
# Este notebook simula la Task 2 que dependería de Task 1
dbutils.widgets.text("process_date", "2026-05-22", "📅 Fecha")
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "🌍 Env")

process_date = dbutils.widgets.get("process_date")
env = dbutils.widgets.get("env")

print(f"🚀 Task 2: Transformación")
print(f"  Fecha: {process_date}")
print(f"  Env: {env}")

In [ ]:
try:
    # Leer datos de Task 1
    input_path = f"dbfs:/user/hive/warehouse/jobs_{env}_bronze_{process_date.replace('-', '')}"
    df_bronze = spark.read.format("delta").load(input_path)
    
    print(f"📊 Leídos: {df_bronze.count()} registros de {input_path}")
    
    # Transformaciones
    df_silver = df_bronze \
        .withColumn("value_rounded", round(col("value"), 2)) \
        .withColumn("value_category",
                    when(col("value") < 250, "Low")
                    .when(col("value") < 750, "Medium")
                    .otherwise("High")) \
        .withColumn("processed_at", current_timestamp())
    
    output_path = f"dbfs:/user/hive/warehouse/jobs_{env}_silver_{process_date.replace('-', '')}"
    
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .save(output_path)
    
    transform_count = df_silver.count()
    print(f"✅ Transformación: {transform_count} registros → {output_path}")
    
    result = {
        "status": "SUCCESS",
        "task": "transform",
        "records_processed": transform_count,
        "output_path": output_path,
        "timestamp": datetime.utcnow().isoformat()
    }
    
    dbutils.notebook.exit(json.dumps(result))
    
except Exception as e:
    error_result = {
        "status": "FAILED",
        "task": "transform",
        "error": str(e),
        "timestamp": datetime.utcnow().isoformat()
    }
    print(f"❌ Error: {e}")
    dbutils.notebook.exit(json.dumps(error_result))

## Notebook 3: Agregación (Task 3)

In [ ]:
dbutils.widgets.text("process_date", "2026-05-22", "📅 Fecha")
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "🌍 Env")

process_date = dbutils.widgets.get("process_date")
env = dbutils.widgets.get("env")

print(f"🚀 Task 3: Agregación")

In [ ]:
try:
    # Leer datos de Task 2
    input_path = f"dbfs:/user/hive/warehouse/jobs_{env}_silver_{process_date.replace('-', '')}"
    df_silver = spark.read.format("delta").load(input_path)
    
    # Agregaciones
    df_gold = df_silver \
        .groupBy("date", "category", "value_category") \
        .agg(
            count("*").alias("num_records"),
            sum("value_rounded").alias("total_value"),
            avg("value_rounded").alias("avg_value"),
            min("value_rounded").alias("min_value"),
            max("value_rounded").alias("max_value")
        )
    
    output_path = f"dbfs:/user/hive/warehouse/jobs_{env}_gold_{process_date.replace('-', '')}"
    
    df_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .save(output_path)
    
    agg_count = df_gold.count()
    print(f"✅ Agregación: {agg_count} grupos → {output_path}")
    
    result = {
        "status": "SUCCESS",
        "task": "aggregate",
        "groups_created": agg_count,
        "output_path": output_path,
        "timestamp": datetime.utcnow().isoformat()
    }
    
    dbutils.notebook.exit(json.dumps(result))
    
except Exception as e:
    error_result = {
        "status": "FAILED",
        "task": "aggregate",
        "error": str(e),
        "timestamp": datetime.utcnow().isoformat()
    }
    print(f"❌ Error: {e}")
    dbutils.notebook.exit(json.dumps(error_result))

## Orquestador - Simula Multi-Task Workflow

In [ ]:
# Este notebook simula la orquestación de las 3 tareas
import json
from datetime import datetime

print("🎯 Iniciando Workflow Multi-Task\n")

# Parámetros
params = {
    "process_date": "2026-05-22",
    "env": "dev",
    "batch_size": "1000"
}

workflow_results = {}

In [ ]:
# Simular Task 1 - Ingest
print("🔹 Ejecutando Task 1: Ingest")
# En un workflow real, usarías: dbutils.notebook.run("Task1_Ingest", 0, params)

# Simular resultado
task1_result = {
    "status": "SUCCESS",
    "task": "ingest",
    "records_ingested": 1000,
    "output_path": "/tmp/jobs/dev/bronze/2026-05-22"
}
workflow_results["task1"] = task1_result
print(f"  ✅ Task 1 completada: {task1_result['records_ingested']} registros")
print()

In [ ]:
# Simular Task 2 - Transform
if workflow_results["task1"]["status"] == "SUCCESS":
    print("🔹 Ejecutando Task 2: Transform")
    
    task2_result = {
        "status": "SUCCESS",
        "task": "transform",
        "records_processed": 1000,
        "output_path": "/tmp/jobs/dev/silver/2026-05-22"
    }
    workflow_results["task2"] = task2_result
    print(f"  ✅ Task 2 completada: {task2_result['records_processed']} registros")
else:
    print("  ⏭️  Task 2 omitida (Task 1 falló)")
print()

In [ ]:
# Simular Task 3 - Aggregate
if workflow_results["task2"]["status"] == "SUCCESS":
    print("🔹 Ejecutando Task 3: Aggregate")
    
    task3_result = {
        "status": "SUCCESS",
        "task": "aggregate",
        "groups_created": 15,
        "output_path": "/tmp/jobs/dev/gold/2026-05-22"
    }
    workflow_results["task3"] = task3_result
    print(f"  ✅ Task 3 completada: {task3_result['groups_created']} grupos")
else:
    print("  ⏭️  Task 3 omitida (Task 2 falló)")
print()

In [ ]:
# Resumen del Workflow
print("\n" + "="*50)
print("📊 RESUMEN DEL WORKFLOW")
print("="*50)

all_success = all(r["status"] == "SUCCESS" for r in workflow_results.values())

if all_success:
    print("\n✅ Workflow completado exitosamente\n")
else:
    print("\n❌ Workflow completado con errores\n")

for task_name, result in workflow_results.items():
    status_icon = "✅" if result["status"] == "SUCCESS" else "❌"
    print(f"{status_icon} {task_name.upper()}: {result['status']}")
    for key, value in result.items():
        if key not in ["status", "task"]:
            print(f"    - {key}: {value}")
    print()

print("="*50)

## Resumen del Laboratorio

### Competencias Adquiridas:

**Parametrización de Notebooks:**
- Uso de dbutils.widgets para recibir parámetros de runtime
- Diseño de notebooks reutilizables para diferentes ambientes (dev/staging/prod)

**Error Handling:**
- Implementación de try-catch con logging detallado
- Retorno de resultados estructurados (JSON) con dbutils.notebook.exit()
- Estrategias de recuperación ante fallos

**Workflows Multi-Task:**
- Orquestación de pipelines con dependencias entre tareas
- Propagación de outputs entre notebooks (Task 1 → Task 2 → Task 3)
- Simulación de workflows de producción (Ingest → Transform → Aggregate)

**Mejores Prácticas:**
- Paths dinámicos por ambiente y fecha para facilitar testing y recovery
- Métricas y timestamps para observabilidad
- Estados de ejecución (SUCCESS/FAILED) para monitoreo

### Próximos Pasos:
- **Lab 06**: Integración y streaming de datos
- Implementación real de Databricks Workflows en la UI
- Configuración de schedules y alertas
